# L2 logistic regression baseline

Our target is self-reported short sleep, defined as fewer than seven hours. This first run uses `C=1` to establish a validation benchmark. Logistic regression estimates how the ten inputs are associated with the odds of short sleep and gives later models a reference for prediction performance.

## 1. Load the cleaned data and define the outcome

Use the cleaned CSV from the data preparation notebook. Its `enough_sleep` column uses 1 for at least seven hours, so we invert it. The ten predictors are categorical, including columns stored as numeric codes.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

cwd = Path.cwd()
data_dir = next(
    (
        folder / "Data_preprocessing"
        for folder in (cwd, *cwd.parents)
        if (folder / "Data_preprocessing" / "brfss_2022_clean.csv").is_file()
    ),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Open the repository in VS Code so the notebook can find Data_preprocessing.")

features = [
    "age_group", "sex", "income_group", "education_level",
    "employment_status", "marital_status", "children_group",
    "physical_activity", "smoking_status", "state",
]
column_types = dict.fromkeys(features, str)
df = pd.read_csv(data_dir / "brfss_2022_clean.csv", dtype=column_types)
assert df["enough_sleep"].isin([0, 1]).all()
assert df["survey_weight"].gt(0).all()
df["short_sleep"] = 1 - df["enough_sleep"]

print(f"Respondents: {len(df):,}")
print(f"Short-sleep respondents: {df['short_sleep'].sum():,}")

W20261115 20:36:05.250730 48483840 cpu_info.cc:239] IOError: sysctlbyname failed for 'hw.l1dcachesize'. Detail: [errno 1] Operation not permitted
W20261115 20:36:05.251530 48483840 cpu_info.cc:239] IOError: sysctlbyname failed for 'hw.l2cachesize'. Detail: [errno 1] Operation not permitted
W20261115 20:36:05.251626 48483840 cpu_info.cc:239] IOError: sysctlbyname failed for 'hw.l3cachesize'. Detail: [errno 1] Operation not permitted
W20261115 20:36:05.251714 48483840 cpu_info.cc:239] IOError: sysctlbyname failed for 'hw.optional.sse4_2'. Detail: [errno 1] Operation not permitted
W20261115 20:36:05.251799 48483840 cpu_info.cc:239] IOError: sysctlbyname failed for 'hw.optional.avx1_0'. Detail: [errno 1] Operation not permitted
W20261115 20:36:05.251883 48483840 cpu_info.cc:239] IOError: sysctlbyname failed for 'hw.optional.avx2_0'. Detail: [errno 1] Operation not permitted
W20261115 20:36:05.251962 48483840 cpu_info.cc:239] IOError: sysctlbyname failed for 'hw.optional.bmi1'. Detail: [err

Respondents: 427,573
Short-sleep respondents: 139,673


## 2. Use the existing train, validation, and test assignments

The cleaned CSV includes the shared `split` column. During preprocessing, each state and primary sampling unit (PSU) combination was assigned to one partition. We use those assignments so this baseline and the tree models are compared on the same respondents. Training rows are used for fitting, validation rows are used for comparing settings, and test rows are reserved for final evaluation.

In [2]:
split_order = ["train", "validation", "test"]

if "split" not in df.columns:
    raise ValueError("Use the updated brfss_2022_clean.csv from the data preparation notebook.")
assert set(df["split"]) == set(split_order), "Each respondent must have a valid split label."

counts = df["split"].value_counts().reindex(split_order)
display(counts.to_frame("respondents"))

train = df.loc[df["split"].eq("train")]
validation = df.loc[df["split"].eq("validation")]
display(
    df.loc[df["split"].isin(["train", "validation"])]
    .groupby("split")["short_sleep"].mean().mul(100).round(3)
    .to_frame("sample short sleep (%)")
)

,respondents
split,
train,256605
validation,85485
test,85483


,sample short sleep (%)
split,
train,32.623
validation,32.871


## 3. Fit an L2 logistic regression pipeline

One-hot encoding gives each category its own indicator. The pipeline learns the categories from training rows and applies the resulting encoding to validation rows. L2 regularization shrinks large coefficients. Smaller `C` means stronger shrinkage. Training survey weights are divided by their mean so they retain their relative weights while averaging one.

In [3]:
train_weights = train["survey_weight"]
model = make_pipeline(
    OneHotEncoder(handle_unknown="ignore", sparse_output=True),
    LogisticRegression(C=1, solver="lbfgs", max_iter=1_000),
)
model.fit(
    train[features],
    train["short_sleep"],
    logisticregression__sample_weight=train_weights / train_weights.mean(),
)
print(f"Encoded features: {len(model.named_steps['onehotencoder'].get_feature_names_out())}")

Encoded features: 113


## 4. Score on the validation set

Survey-weighted average precision (AP) is our primary metric. It summarizes precision across recall levels when we rank respondents by their predicted probability of short sleep. The constant-score reference has AP equal to the validation set's weighted prevalence. ROC-AUC summarizes how well scores rank short sleepers ahead of other respondents.

In [4]:
val_weights = validation["survey_weight"]
y_val = validation["short_sleep"]
scores = model.predict_proba(validation[features])[:, 1]
weighted_prevalence = np.average(y_val, weights=val_weights)

metrics = pd.DataFrame(
    [
        [weighted_prevalence, 0.5],
        [
            average_precision_score(y_val, scores, sample_weight=val_weights),
            roc_auc_score(y_val, scores, sample_weight=val_weights),
        ],
    ],
    index=["Constant score", "L2 logistic regression"],
    columns=["Weighted AP", "Weighted ROC-AUC"],
)
display(metrics.round(3))

,Weighted AP,Weighted ROC-AUC
Constant score,0.363,0.500
L2 logistic regression,0.464,0.614


## 5. Interpret selected category contrasts

For each contrast, subtract the reference category's coefficient from the comparison category's coefficient in the same feature. Exponentiating that difference gives an odds ratio, with the other modeled features held fixed. An odds ratio above one corresponds to higher modeled odds of short sleep. These estimates describe conditional associations in the fitted model.

In [5]:
encoder = model.named_steps["onehotencoder"]
logistic = model.named_steps["logisticregression"]
coefficients = pd.Series(logistic.coef_[0], index=encoder.get_feature_names_out(features))

contrasts = [
    ("physical_activity", "2", "1", "No leisure-time activity vs. any"),
    ("income_group", "1", "7", "Household income under $10k vs. $50k to under $75k"),
    ("employment_status", "8", "1", "Unable to work vs. employed for wages"),
]
odds_ratios = pd.DataFrame(
    [
        {
            "Contrast": description,
            "Odds ratio": np.exp(coefficients[f"{feature}_{level}"] - coefficients[f"{feature}_{reference}"]),
        }
        for feature, level, reference, description in contrasts
    ]
)
display(odds_ratios.round(3))

,Contrast,Odds ratio
0,No leisure-time activity vs. any,1.235
1,Household income under $10k vs. $50k to under ...,0.992
2,Unable to work vs. employed for wages,1.251


## Next step

Once the first run works, tune `C` using weighted AP on the validation set. Keep the test set reserved while choosing model settings.